In [ ]:
import os.path
import pandas as pd
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

def main():
    creds = None
    if os.path.exists("token.json"):
        creds = Credentials.from_authorized_user_file("token.json", SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                os.path.expanduser("C:\\Users\\Melissa\\Downloads\\credentials.json"),
                SCOPES,
            )
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as token:
            token.write(creds.to_json())
    try:
        service = build("gmail", "v1", credentials=creds)
        results = service.users().messages().list(userId="me", labelIds=["INBOX"],maxResults=10).execute()
        messages = results.get("messages", [])
        if not messages:
            print("No messages found.")
            return
        print("Messages:")
        messages_data = []
        for message in messages:
            msg = service.users().messages().get(userId="me", id=message["id"]).execute()
            #Store the message data in a dataset
#            messages_data.append({
#            "id": message["id"],
#            "Snippet": msg.get("snippet", "")
#            "Subject: {msg["headers"][0]["value"] if "headers" in msg and len(msg["headers"]) > 0 else "N/A"}'),
#            "labelIds: {msg["labelIds"] if "labelIds" in msg else "N/A"}') 
#            })    
            #Print the data on screen
            print(f'Message ID: {message["id"]}')
            print(f'  Snippet: {msg["snippet"]}')
            print(f'  Subject: {msg["payload"]["headers"][0]["value"] if "headers" in msg and len(msg["headers"]) > 0 else "N/A"}')
            print(f'  labelIds: {msg["labelIds"] if "labelIds" in msg else "N/A"}')
        df = pd.DataFrame(messages_data)    
    except HttpError as error:
        print(f"An error occurred: {error}")

if __name__ == "__main__":
    main()
